<a href="https://colab.research.google.com/github/ssurapaneni34/702project/blob/main/agent_single_v6_1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🧠 ICU Clinical Grouping — Single Agent Approach (v3)

**Simplified Architecture:**
- LSTM predicts top-3 clinical groupings (used only to narrow the candidate set)
- **ONE agent** is shown the 3 reference cards + patient data and must choose
- The agent is **blind** to the LSTM: no probabilities, no ranking — candidates are shuffled
- Agent reports confidence qualitatively: **low / medium / high**

**Why blind the agent:**
- Prevents anchoring to the LSTM's top pick
- Forces the decision to rest on the clinical evidence, not the model's ordering
- Makes disagreement a genuine clinical signal rather than a confidence-threshold effect

**Trade-off:**
- Longer prompt (3 reference cards)
- Less structured reasoning breakdown

## 1. Setup & Configuration

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# SETUP
# ══════════════════════════════════════════════════════════════════════════════
!pip install -q google-generativeai scikit-learn joblib tqdm

from google.colab import drive, userdata
drive.mount('/content/drive')

print('Setup complete ✓')

Mounted at /content/drive
Setup complete ✓


In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# CONFIGURATION
# ══════════════════════════════════════════════════════════════════════════════
import os

# ── File Paths ────────────────────────────────────────────────────────────────
BASE_DIR = '/content/drive/MyDrive/702 project'
LSTM_OUTPUT_DIR = f'{BASE_DIR}/lstm_outputs_v2'

CARDS_PATH = f'{BASE_DIR}/patient_cards_v6_grouped.json'
MODEL_PATH = f'{LSTM_OUTPUT_DIR}/best_model.pt'
SCALER_PATH = f'{LSTM_OUTPUT_DIR}/feature_scaler.pkl'
CONFIG_PATH = f'{LSTM_OUTPUT_DIR}/model_config.json'

# Output directory
AGENT_OUTPUT_DIR = f'{BASE_DIR}/agent_single_v3_results'
os.makedirs(AGENT_OUTPUT_DIR, exist_ok=True)

# ── Pilot Configuration ───────────────────────────────────────────────────────
PILOT_N_CASES = 5
TOP_K = 3                       # Only top-3 (simpler for single agent)
RANDOM_SEED = 42

# ── LLM Configuration ─────────────────────────────────────────────────────────
GEMINI_MODEL = 'gemini-2.5-flash-lite'
REQUESTS_PER_MINUTE = 10
RETRY_ATTEMPTS = 5
RETRY_DELAY_BASE = 3

print(f'Configuration:')
print(f'  Cases         : {PILOT_N_CASES}')
print(f'  Top-K         : {TOP_K}')
print(f'  LLM calls     : {PILOT_N_CASES} (1 per case!)')
print(f'  Model         : {GEMINI_MODEL}')
print(f'  Output        : {AGENT_OUTPUT_DIR}')

Configuration:
  Cases         : 5
  Top-K         : 3
  LLM calls     : 5 (1 per case!)
  Model         : gemini-2.5-flash-lite
  Output        : /content/drive/MyDrive/702 project/agent_single_v3_results


In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# API KEY
# ══════════════════════════════════════════════════════════════════════════════
import google.generativeai as genai
try:
    GOOGLE_API_KEY = userdata.get('GOOGLE_API_KEY')
    print('✓ API key loaded from Colab Secrets')
except:
    raise ValueError('Add GEMINI_API_KEY to Colab Secrets (sidebar → 🔑)')

genai.configure(api_key=GOOGLE_API_KEY)
print('✓ Gemini API configured')

✓ API key loaded from Colab Secrets
✓ Gemini API configured


## 2. Load Data & Model

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# LOAD ALL DATA
# ══════════════════════════════════════════════════════════════════════════════
import json
import numpy as np
import random
import torch
import torch.nn as nn
import joblib
from collections import defaultdict, Counter

# Load patient cards
print('Loading patient cards...')
with open(CARDS_PATH) as f:
    data = json.load(f)

all_cards = data['patients']
label_map = data['label_map']
feature_names = data['feature_names']
n_features = data['n_features']
n_timesteps = data['n_timesteps']
num_classes = len(label_map)
inv_label_map = {v: k for k, v in label_map.items()}

val_idx = list(range(0, n_features, 2))
mask_idx = list(range(1, n_features, 2))

print(f'  Patients: {len(all_cards):,} | Classes: {num_classes}')

# Load model config
with open(CONFIG_PATH) as f:
    config = json.load(f)

# Load scaler
scaler = joblib.load(SCALER_PATH)
print('✓ All data loaded')

Loading patient cards...
  Patients: 28,467 | Classes: 21
✓ All data loaded


In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# LOAD LSTM MODEL
# ══════════════════════════════════════════════════════════════════════════════
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {device}')

class LSTMWithAttention(nn.Module):
    def __init__(self, input_dim, hidden_dim, num_layers, num_classes,
                 dropout=0.3, num_heads=4):
        super().__init__()
        self.hidden_dim = hidden_dim
        self.num_layers = num_layers
        self.lstm = nn.LSTM(
            input_size=input_dim, hidden_size=hidden_dim, num_layers=num_layers,
            batch_first=True, dropout=dropout if num_layers > 1 else 0, bidirectional=True
        )
        self.attention = nn.MultiheadAttention(
            embed_dim=hidden_dim * 2, num_heads=num_heads, dropout=dropout, batch_first=True
        )
        self.layer_norm = nn.LayerNorm(hidden_dim * 2)
        self.dropout = nn.Dropout(dropout)
        self.fc1 = nn.Linear(hidden_dim * 2, hidden_dim)
        self.fc2 = nn.Linear(hidden_dim, num_classes)

    def forward(self, x, return_attention=False):
        lstm_out, _ = self.lstm(x)
        attn_out, attn_weights = self.attention(lstm_out, lstm_out, lstm_out, need_weights=True)
        attn_out = self.layer_norm(lstm_out + attn_out)
        pooled = attn_out.mean(dim=1)
        out = self.dropout(pooled)
        out = torch.relu(self.fc1(out))
        out = self.dropout(out)
        logits = self.fc2(out)
        return (logits, attn_weights) if return_attention else logits

model = LSTMWithAttention(
    input_dim=config['input_dim'], hidden_dim=config['hidden_dim'],
    num_layers=config['num_layers'], num_classes=config['num_classes'],
    dropout=config['dropout'], num_heads=config['num_heads']
).to(device)

model.load_state_dict(torch.load(MODEL_PATH, map_location=device))
model.eval()
print(f'✓ LSTM loaded ({sum(p.numel() for p in model.parameters()):,} params)')

Device: cuda
✓ LSTM loaded (3,572,245 params)


In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# CREATE TEST SPLIT & PILOT SAMPLE
# ══════════════════════════════════════════════════════════════════════════════
TRAIN_RATIO, VAL_RATIO = 0.70, 0.15

random.seed(42)
np.random.seed(42)

patient_to_cards = defaultdict(list)
for i, card in enumerate(all_cards):
    patient_to_cards[card['patient_id']].append(i)

patient_ids = list(patient_to_cards.keys())
random.shuffle(patient_ids)

n_train = int(len(patient_ids) * TRAIN_RATIO)
n_val = int(len(patient_ids) * VAL_RATIO)
test_pids = set(patient_ids[n_train + n_val:])
test_cards = [all_cards[i] for pid in test_pids for i in patient_to_cards[pid]]

# Stratified pilot sample
random.seed(RANDOM_SEED)
label_to_cards = defaultdict(list)
for card in test_cards:
    label_to_cards[card['label']].append(card)

label_counts = Counter(c['label'] for c in test_cards)
total = sum(label_counts.values())

pilot_cards = []
for label, count in sorted(label_counts.items(), key=lambda x: -x[1]):
    n_sample = max(1, int(PILOT_N_CASES * count / total))
    n_sample = min(n_sample, len(label_to_cards[label]))
    pilot_cards.extend(random.sample(label_to_cards[label], n_sample))

random.shuffle(pilot_cards)
pilot_cards = pilot_cards[:PILOT_N_CASES]

print(f'Test set: {len(test_cards):,} stays')
print(f'Pilot sample: {len(pilot_cards)} cases')

Test set: 4,336 stays
Pilot sample: 5 cases


## 3. Define Single Agent System

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# PREPROCESSING
# ══════════════════════════════════════════════════════════════════════════════

def normalize_time_series(card):
    """Apply scaler to patient time series."""
    X = np.array(card['time_series'], dtype=np.float32)
    X_vals = X[:, val_idx]
    X_vals_scaled = scaler.transform(X_vals)
    X[:, val_idx] = X_vals_scaled.astype(np.float32)
    return X


def get_lstm_predictions(card, top_k=3):
    """Get LSTM top-K predictions."""
    X = normalize_time_series(card)
    X_tensor = torch.from_numpy(X).unsqueeze(0).to(device)

    with torch.no_grad():
        logits = model(X_tensor)
        probs = torch.softmax(logits, dim=-1).squeeze(0)

    top_k_idx = probs.argsort(descending=True)[:top_k]
    top_k_groups = [inv_label_map[i.item()] for i in top_k_idx]
    top_k_probs = [probs[i].item() for i in top_k_idx]

    return {
        'top_k_codes': top_k_groups,
        'top_k_probs': top_k_probs,
        'lstm_prediction': top_k_groups[0],
        'lstm_confidence': top_k_probs[0]
    }

# Test
test_pred = get_lstm_predictions(pilot_cards[0], TOP_K)
print(f'✓ LSTM test: {test_pred["lstm_prediction"]} ({test_pred["lstm_confidence"]:.1%})')

✓ LSTM test: TOXIC_METABOLIC (38.6%)


In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# FORMATTERS
# ══════════════════════════════════════════════════════════════════════════════

def format_clinical_summary(summary: dict) -> str:
    """Format clinical summary for prompt."""
    if not summary:
        return "Clinical summary not available"

    lines = []

    if 'vitals' in summary:
        lines.append("VITAL SIGNS (Last 12 Hours):")
        for name, stats in summary['vitals'].items():
            if isinstance(stats, dict) and 'mean' in stats:
                lines.append(
                    f"  {name.replace('_', ' ').title()}: "
                    f"mean={stats['mean']:.1f}, range=[{stats['min']:.1f}–{stats['max']:.1f}], "
                    f"last={stats['last']:.1f} ({stats.get('trend', 'N/A')})"
                )

    if 'labs' in summary:
        lines.append("\nLABORATORY VALUES:")
        for name, stats in summary['labs'].items():
            if isinstance(stats, dict) and 'mean' in stats:
                lines.append(
                    f"  {name.replace('_', ' ').title()}: "
                    f"mean={stats['mean']:.2f}, last={stats['last']:.2f} ({stats.get('trend', 'N/A')})"
                )

    if 'interventions' in summary:
        iv = summary['interventions']
        lines.append("\nINTERVENTIONS:")
        if iv.get('active_drugs'):
            lines.append(f"  Medications: {', '.join(iv['active_drugs'])}")
        if iv.get('active_procedures'):
            lines.append(f"  Procedures: {', '.join(iv['active_procedures'])}")

    return '\n'.join(lines) if lines else "Clinical summary not available"


print('✓ Formatters defined')

✓ Formatters defined


In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# BUILD AGENT-SAFE PATIENT CARD
# ══════════════════════════════════════════════════════════════════════════════
# The patient cards from preprocessing include `icd_code` and `clinical_group`
# at the top level, plus `procedures` and `drugs` lists covering the entire
# hospital admission (which temporally extends past the 12-hour ICU window
# and so encodes post-diagnosis information). Even though the current
# formatter only reads `clinical_summary`, we build a scrubbed copy of the
# card before calling the LLM so the answer cannot leak through any future
# change, logging, or serialization.
#
# The ground-truth `label` (integer), `clinical_group`, and `icd_code` are
# kept on the ORIGINAL card for validation downstream.

# Fields that reveal the diagnosis — stripped before the card is shown to any agent
LEAKY_CARD_FIELDS = {
    'icd_code',         # raw ICD-10 code, e.g. "J189"
    'clinical_group',   # text answer, e.g. "PNEUMONIA"
    'label',            # integer index into label_map — still the answer
    'procedures',       # whole-admission ICD-PCS codes (temporal leak)
    'drugs',            # whole-admission prescriptions (temporal leak)
    'conditions',       # reserved PyHealth field; may contain dx history
}


def build_clinical_summary_for_agent(card: dict) -> dict:
    """Return a shallow copy of the card with all diagnosis-revealing fields removed.

    Keeps what the agent legitimately needs:
      - patient_id, stay_id, visit_id (identifiers; not diagnostic)
      - time_series, feature_names  (raw numeric signal)
      - clinical_summary            (12hr-window vitals/labs/interventions)

    Removes:
      - icd_code, clinical_group, label  (the answer)
      - procedures, drugs                (whole-admission → temporal leak)
      - conditions                       (reserved field that may carry dx history)
    """
    safe = {k: v for k, v in card.items() if k not in LEAKY_CARD_FIELDS}

    # Defensive pass on clinical_summary itself: even though its normal schema
    # is {vitals, labs, interventions, observation_hours, pct_hours_monitored},
    # scrub any unexpected keys whose names hint at diagnosis.
    cs = safe.get('clinical_summary')
    if isinstance(cs, dict):
        diag_hint_keys = [
            k for k in cs.keys()
            if any(tok in k.lower()
                   for tok in ('diag', 'icd', 'group', 'label', 'condition',
                               'problem', 'impression'))
        ]
        if diag_hint_keys:
            cs = {k: v for k, v in cs.items() if k not in diag_hint_keys}
            safe['clinical_summary'] = cs

    return safe


# Sanity-check on pilot_cards[0] — confirm the stripped card has no leaks
_probe = build_clinical_summary_for_agent(pilot_cards[0])
_leaks_present = LEAKY_CARD_FIELDS & set(_probe.keys())
assert not _leaks_present, f'Leak check failed: {_leaks_present} still in stripped card'
print(f'✓ build_clinical_summary_for_agent() defined')
print(f'  Original card keys   : {sorted(pilot_cards[0].keys())}')
print(f'  Agent-safe card keys : {sorted(_probe.keys())}')
print(f'  Stripped             : {sorted(set(pilot_cards[0].keys()) - set(_probe.keys()))}')


✓ build_clinical_summary_for_agent() defined
  Original card keys   : ['clinical_group', 'clinical_summary', 'conditions', 'drugs', 'feature_names', 'icd_code', 'label', 'patient_id', 'procedures', 'stay_id', 'time_series', 'visit_id']
  Agent-safe card keys : ['clinical_summary', 'feature_names', 'patient_id', 'stay_id', 'time_series', 'visit_id']
  Stripped             : ['clinical_group', 'conditions', 'drugs', 'icd_code', 'label', 'procedures']


In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# INSPECT AGENT-SAFE CARDS (pre-flight audit)
# ══════════════════════════════════════════════════════════════════════════════
# Before running any LLM calls, dump exactly what the agent would see for the
# first 5 pilot cards. Nothing here is shown to the LLM — this is for you.
import pprint, json as _json

N_INSPECT = 5
pp = pprint.PrettyPrinter(width=100, depth=4, sort_dicts=False)

for inspect_i, inspect_card in enumerate(pilot_cards[:N_INSPECT]):
    print('\n' + '█' * 78)
    print(f'█ AGENT-SAFE CARD PREVIEW  [{inspect_i + 1}/{N_INSPECT}]')
    print('█' * 78)

    # Ground truth — printed HERE for your reference only, never sent to the LLM
    _true = inv_label_map[inspect_card['label']]
    print(f'\n➤ GROUND TRUTH (not shown to agent):  {_true}')
    print(f'  (integer label {inspect_card["label"]}, patient {inspect_card["patient_id"]})')

    # 1. Raw agent-safe card dict
    safe = build_clinical_summary_for_agent(inspect_card)
    print('\n── 1. AGENT-SAFE CARD (top-level keys) ──')
    print(f'  Keys: {sorted(safe.keys())}')
    print(f'  Stripped: {sorted(set(inspect_card.keys()) - set(safe.keys()))}')

    # 2. Full clinical_summary dict
    print('\n── 2. FULL clinical_summary DICT (what the formatter reads from) ──')
    pp.pprint(safe.get('clinical_summary', {}))

    # 3. Formatted clinical summary string — the literal text going into the prompt
    print('\n── 3. FORMATTED CLINICAL SUMMARY (exact text in the prompt) ──')
    formatted = format_clinical_summary(safe.get('clinical_summary', {}))
    print(formatted)

    # 4. Example shuffle of top-K LSTM candidates for this patient
    _lstm = get_lstm_predictions(safe, TOP_K)
    _rng = random.Random(
        str(safe.get('patient_id', '')) + '|' + str(safe.get('stay_id', ''))
    )
    _shuffled = list(_lstm['top_k_codes'])
    _rng.shuffle(_shuffled)
    print('\n── 4. LSTM TOP-K AND SHUFFLED ORDER ──')
    print(f'  LSTM top-{TOP_K} (with probs, for your reference only):')
    for _rank, (_c, _p) in enumerate(zip(_lstm["top_k_codes"], _lstm["top_k_probs"])):
        _mark = "✓" if _c == _true else " "
        print(f'    {_rank+1}. {_c:<25} {_p:>6.1%} {_mark}')
    print(f'  Shuffled order shown to agent (no probs, no ranking):')
    for _c in _shuffled:
        print(f'    - {_c}')

print('\n' + '█' * 78)
print(f'█ End of agent-safe card preview ({N_INSPECT} cards)')
print('█' * 78)



██████████████████████████████████████████████████████████████████████████████
█ AGENT-SAFE CARD PREVIEW  [1/5]
██████████████████████████████████████████████████████████████████████████████

➤ GROUND TRUTH (not shown to agent):  LIVER_FAILURE
  (integer label 8, patient 17725178)

── 1. AGENT-SAFE CARD (top-level keys) ──
  Keys: ['clinical_summary', 'feature_names', 'patient_id', 'stay_id', 'time_series', 'visit_id']
  Stripped: ['clinical_group', 'conditions', 'drugs', 'icd_code', 'label', 'procedures']

── 2. FULL clinical_summary DICT (what the formatter reads from) ──
{'vitals': {'heart_rate': {'mean': 110.18,
                           'min': 102.0,
                           'max': 126.0,
                           'last': 119.0,
                           'first': 111.0,
                           'trend': 'stable',
                           'n_observations': 11},
            'respiratory_rate': {'mean': 23.55,
                                 'min': 17.0,
                  

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# SINGLE AGENT PROMPT
# ══════════════════════════════════════════════════════════════════════════════

SINGLE_AGENT_PROMPT = """
You are an expert ICU physician. You are given patient data and {top_k} possible diagnoses to consider. Your task is to determine which diagnosis best fits the clinical picture.

## PATIENT DATA (Last 12 Hours in ICU)
{clinical_summary}

## CANDIDATE DIAGNOSES
Consider these {top_k} possibilities (listed in no particular order):
{candidates_list}


## YOUR TASK
You are working with INCOMPLETE ICU data. Not all clinically relevant labs will be present. Your goal is to find the diagnosis most consistent with available evidence, explicitly accounting for what is and is not measured. A diagnosis should not be dismissed solely because its defining marker was not collected.


Respond with ONLY valid JSON:
{{
  "chosen_diagnosis": "DIAGNOSIS_NAME",
  "reasoning": "2-3 sentence explanation of why this diagnosis fits best",
  "supporting_evidence": ["key finding 1", "key finding 2", "key finding 3"],
  "ruled_out": [
    {{"diagnosis": "OTHER_DIAGNOSIS_1", "reason": "why ruled out"}},
    {{"diagnosis": "OTHER_DIAGNOSIS_2", "reason": "why ruled out"}}
  ]
}}

Your chosen_diagnosis MUST be one of the {top_k} candidates listed above.
"""

print('✓ Single agent prompt defined')


✓ Single agent prompt defined


In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# LLM WRAPPER
# ══════════════════════════════════════════════════════════════════════════════
import time
import re
from google.api_core import exceptions as google_exceptions

gemini_model = genai.GenerativeModel(GEMINI_MODEL)
call_times = []

def rate_limit_wait():
    global call_times
    now = time.time()
    call_times = [t for t in call_times if now - t < 60]
    if len(call_times) >= REQUESTS_PER_MINUTE:
        wait_time = 60 - (now - call_times[0]) + 2.0
        if wait_time > 0:
            print(f'    ⏳ Rate limit: waiting {wait_time:.1f}s...')
            time.sleep(wait_time)
    call_times.append(time.time())


def call_llm(prompt: str) -> dict:
    """Call Gemini with retry logic."""
    for attempt in range(RETRY_ATTEMPTS):
        rate_limit_wait()
        try:
            response = gemini_model.generate_content(
                prompt,
                generation_config=genai.GenerationConfig(
                    temperature=0.2,  # Lower temp for more consistent reasoning
                    max_output_tokens=1000,
                )
            )
            if not response.text:
                raise ValueError("Empty response")

            json_match = re.search(r'\{[\s\S]*\}', response.text.strip())
            if json_match:
                return json.loads(json_match.group(0))
            raise ValueError("No JSON found")

        except google_exceptions.ResourceExhausted as e:
            wait = 30
            match = re.search(r'retry in ([\d.]+)s', str(e).lower())
            if match:
                wait = float(match.group(1)) + 2
            print(f'    ⚠ Rate limited. Waiting {wait:.0f}s...')
            time.sleep(wait)

        except (google_exceptions.ServiceUnavailable, google_exceptions.InternalServerError) as e:
            wait = RETRY_DELAY_BASE * (2 ** attempt)
            print(f'    ⚠ Server error. Waiting {wait:.0f}s... (attempt {attempt+1})')
            time.sleep(wait)

        except Exception as e:
            if attempt < RETRY_ATTEMPTS - 1:
                print(f'    ⚠ {type(e).__name__}: {str(e)[:50]}. Retrying...')
                time.sleep(2)

    return {"error": "Failed after retries", "chosen_diagnosis": None}

print('✓ LLM wrapper ready')

✓ LLM wrapper ready


In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# SINGLE AGENT DIAGNOSIS FUNCTION
# ══════════════════════════════════════════════════════════════════════════════

# VALID_CONFIDENCE_LEVELS = {'low', 'medium', 'high'}


# def _normalize_confidence(raw):
#     """Coerce model output to one of low/medium/high. Accepts strings or numbers."""
#     if isinstance(raw, str):
#         v = raw.strip().lower()
#         if v in VALID_CONFIDENCE_LEVELS:
#             return v
#         if v in {'lo', 'weak'}:
#             return 'low'
#         if v in {'med', 'mid', 'moderate'}:
#             return 'medium'
#         if v in {'hi', 'strong'}:
#             return 'high'
#     if isinstance(raw, (int, float)):
#         if raw < 40:
#             return 'low'
#         if raw < 75:
#             return 'medium'
#         return 'high'
#     return 'low'


def diagnose_single_agent(patient_card: dict, verbose: bool = False) -> dict:
    """Run single-agent diagnosis on one patient. Agent is blind to LSTM ordering/confidence
    AND to the ground-truth fields on the original card.

    When verbose=True, every step is printed in full: the stripped card, the LSTM output,
    the shuffle, the complete prompt sent to the LLM, the raw agent response, and each
    parsed reasoning field.
    """

    def _log(title, body=None):
        if not verbose:
            return
        print(f'\n── {title} ──')
        if body is not None:
            print(body)

    # Build agent-safe view of the card. Everything the LLM sees comes from here.
    clinical_summary_for_agent = build_clinical_summary_for_agent(patient_card)
    _log('STEP A: agent-safe card (top-level keys)',
         f'  keys: {sorted(clinical_summary_for_agent.keys())}')

    # 1. LSTM top-K — uses the stripped card (only needs time_series)
    lstm_output = get_lstm_predictions(clinical_summary_for_agent, TOP_K)
    if verbose:
        print('\n── STEP B: LSTM top-K (for your reference — NOT shown to agent) ──')
        for _rank, (_c, _p) in enumerate(zip(lstm_output['top_k_codes'], lstm_output['top_k_probs'])):
            print(f'  {_rank+1}. {_c:<25} {_p:>6.1%}')

    # 2. Shuffle candidates (seeded per patient for reproducibility)
    shuffle_rng = random.Random(
        str(clinical_summary_for_agent.get('patient_id', '')) + '|' +
        str(clinical_summary_for_agent.get('stay_id', ''))
    )
    shuffled_codes = list(lstm_output['top_k_codes'])
    shuffle_rng.shuffle(shuffled_codes)
    _log('STEP C: shuffled candidate order (what the agent sees)',
         '\n'.join(f'  - {c}' for c in shuffled_codes))

    # 3. Format candidates as a plain list, NO probabilities, NO ranking
    candidates_str = "\n".join([f"- {code}" for code in shuffled_codes])

    # 5. Build prompt
    formatted_summary = format_clinical_summary(
        clinical_summary_for_agent.get('clinical_summary', {})
    )
    _log('STEP D: formatted clinical summary (exact text in prompt)', formatted_summary)

    prompt = SINGLE_AGENT_PROMPT.format(
        top_k=TOP_K,
        clinical_summary=formatted_summary,
        candidates_list=candidates_str,
    )
    _log('STEP F: FULL PROMPT SENT TO LLM', prompt)

    # 6. Call LLM
    agent_output = call_llm(prompt)
    if verbose:
        print('\n── STEP G: RAW LLM RESPONSE (parsed JSON) ──')
        try:
            print(json.dumps(agent_output, indent=2, ensure_ascii=False))
        except (TypeError, ValueError):
            print(repr(agent_output))

    # 7. Extract prediction
    raw_pred = agent_output.get('chosen_diagnosis') or ''
    norm = raw_pred.strip().strip('"\'.,').upper()
    lookup = {c.strip().upper(): c for c in lstm_output['top_k_codes']}
    agent_pred = lookup.get(norm)

    # agent_pred = agent_output.get('chosen_diagnosis')
    print(agent_pred)

    if agent_pred not in lstm_output['top_k_codes']:
        if verbose:
            print(f'\n  ⚠ chosen_diagnosis {agent_pred!r} not in top-K candidates; '
                  f'falling back to RANDOM WORD: WHITEBOARD JELLYFISH')
        agent_pred = 'WHITEBOARD JELLYFISH'

    # 8. Normalize confidence

    if verbose:
        print('\n── STEP H: AGENT REASONING (parsed fields) ──')
        print(f'  chosen_diagnosis : {agent_pred}')
        _reasoning = agent_output.get('reasoning')
        if _reasoning:
            print(f'  reasoning        :')
            for line in str(_reasoning).splitlines() or [str(_reasoning)]:
                print(f'      {line}')
        _support = agent_output.get('supporting_evidence') or []
        if _support:
            print(f'  supporting_evidence:')
            for item in _support:
                print(f'      • {item}')
        _ruled = agent_output.get('ruled_out') or []
        if _ruled:
            print(f'  ruled_out:')
            for item in _ruled:
                if isinstance(item, dict):
                    print(f'      • {item.get("diagnosis", "?")}: {item.get("reason", "")}')
                else:
                    print(f'      • {item}')

    # 9. Read ground truth from the ORIGINAL card (validation only, never sent to LLM)
    true_label = inv_label_map[patient_card['label']]

    return {
        'patient_id': patient_card.get('patient_id'),
        'stay_id': patient_card.get('stay_id'),
        'true_label': true_label,
        'lstm_output': lstm_output,
        'candidates_shown_order': shuffled_codes,
        'agent_output': agent_output,
        'agent_prediction': agent_pred,
        'agent_disagrees': agent_pred != lstm_output['lstm_prediction'],
        'lstm_correct': lstm_output['lstm_prediction'] == true_label,
        'agent_correct': agent_pred == true_label,
    }

print('✓ Single agent system ready (agent sees only the stripped card; verbose mode available)')


✓ Single agent system ready (agent sees only the stripped card; verbose mode available)


## 4. Run Pilot

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# RUN PILOT  (verbose: full per-case logging)
# ══════════════════════════════════════════════════════════════════════════════
from tqdm.notebook import tqdm
import datetime

print(f'Starting single-agent pilot: {len(pilot_cards)} cases')
print(f'LLM calls: {len(pilot_cards)} (only 1 per case!)')
print(f'Model: {GEMINI_MODEL}')
print(f'Verbose: True  —  every step of every case will be printed in full.')
print('=' * 78)

results = []
start_time = datetime.datetime.now()

for i, card in enumerate(tqdm(pilot_cards, desc="Processing")):
    true_label = inv_label_map[card['label']]

    print('\n' + '═' * 78)
    print(f'CASE [{i+1}/{len(pilot_cards)}]  Patient {card["patient_id"]}  |  True: {true_label}')
    print('═' * 78)

    try:
        result = diagnose_single_agent(card, verbose=True)
        results.append(result)

        # Outcome line
        lstm_mark  = '✓' if result['lstm_correct']  else '✗'
        agent_mark = '✓' if result['agent_correct'] else '✗'
        disagree   = '  🔄 disagreed' if result['agent_disagrees'] else ''
        print('\n── OUTCOME ──')
        print(f'  LSTM pick  : {result["lstm_output"]["lstm_prediction"]} {lstm_mark}')
        print(f'  Agent pick : {result["agent_prediction"]} {agent_mark}{disagree}')

    except Exception as e:
        print(f'\n  ERROR: {type(e).__name__}: {e}')
        results.append({'patient_id': card.get('patient_id'), 'error': str(e)})

elapsed = datetime.datetime.now() - start_time
print('\n' + '=' * 78)
print(f'Pilot complete — elapsed: {elapsed}')
print(f'Valid: {sum(1 for r in results if "error" not in r)}/{len(results)}')


Starting single-agent pilot: 5 cases
LLM calls: 5 (only 1 per case!)
Model: gemini-2.5-flash-lite
Verbose: True  —  every step of every case will be printed in full.


Processing:   0%|          | 0/5 [00:00<?, ?it/s]


══════════════════════════════════════════════════════════════════════════════
CASE [1/5]  Patient 17725178  |  True: LIVER_FAILURE
══════════════════════════════════════════════════════════════════════════════

── STEP A: agent-safe card (top-level keys) ──
  keys: ['clinical_summary', 'feature_names', 'patient_id', 'stay_id', 'time_series', 'visit_id']

── STEP B: LSTM top-K (for your reference — NOT shown to agent) ──
  1. TOXIC_METABOLIC            38.6%
  2. SEPSIS                     30.5%
  3. ACUTE_MI                   11.5%

── STEP C: shuffled candidate order (what the agent sees) ──
  - SEPSIS
  - TOXIC_METABOLIC
  - ACUTE_MI

── STEP D: formatted clinical summary (exact text in prompt) ──
VITAL SIGNS (Last 12 Hours):
  Heart Rate: mean=110.2, range=[102.0–126.0], last=119.0 (stable)
  Respiratory Rate: mean=23.6, range=[17.0–59.0], last=21.0 (worsening)
  O2 Saturation Pulseoxymetry: mean=96.6, range=[87.0–100.0], last=100.0 (stable)
  Non Invasive Blood Pressure Systolic:

    ⚠ Forbidden: 403 POST https://generativelanguage.googleapis.com. Retrying...


    ⚠ Forbidden: 403 POST https://generativelanguage.googleapis.com. Retrying...


...

    ⚠ Forbidden: 403 POST https://generativelanguage.googleapis.com. Retrying...


    ⚠ Forbidden: 403 POST https://generativelanguage.googleapis.com. Retrying...


    ⚠ Forbidden: 403 POST https://generativelanguage.googleapis.com. Retrying...



── STEP G: RAW LLM RESPONSE (parsed JSON) ──
{
  "error": "Failed after retries",
  "chosen_diagnosis": null
}
None

  ⚠ chosen_diagnosis None not in top-K candidates; falling back to RANDOM WORD: WHITEBOARD JELLYFISH

── STEP H: AGENT REASONING (parsed fields) ──
  chosen_diagnosis : WHITEBOARD JELLYFISH

── OUTCOME ──
  LSTM pick  : CORONARY_ARTERY_DZ ✓
  Agent pick : WHITEBOARD JELLYFISH ✗  🔄 disagreed

══════════════════════════════════════════════════════════════════════════════
CASE [3/5]  Patient 14098557  |  True: AFIB_ARRHYTHMIA
══════════════════════════════════════════════════════════════════════════════

── STEP A: agent-safe card (top-level keys) ──
  keys: ['clinical_summary', 'feature_names', 'patient_id', 'stay_id', 'time_series', 'visit_id']

── STEP B: LSTM top-K (for your reference — NOT shown to agent) ──
  1. SEPSIS                     33.1%
  2. AFIB_ARRHYTHMIA            16.5%
  3. TOXIC_METABOLIC             9.8%

── STEP C: shuffled candidate order (what the a

KeyboardInterrupt: 

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# INSPECT A CASE
# ══════════════════════════════════════════════════════════════════════════════

def inspect_case(result):
    print('═' * 70)
    print(f"PATIENT: {result['patient_id']}")
    print('═' * 70)

    print(f"\n🎯 TRUE: {result['true_label']}")

    # LSTM probs are shown here for OUR analysis only; the agent never saw them.
    print(f"\n📊 LSTM TOP-{TOP_K} (not shown to agent):")
    for i, (code, prob) in enumerate(zip(
        result['lstm_output']['top_k_codes'],
        result['lstm_output']['top_k_probs']
    )):
        mark = '✓' if code == result['true_label'] else ' '
        print(f"   {i+1}. {code:<25} {prob:>6.1%} {mark}")

    # What the agent actually saw (shuffled, unlabeled order)
    if 'candidates_shown_order' in result:
        print(f"\n   Candidates shown to agent (shuffled, no probs):")
        for i, code in enumerate(result['candidates_shown_order']):
            print(f"     - {code}")

    print(f"\n🤖 AGENT DECISION:")
    print(f"   Chose: {result['agent_prediction']}")

    ao = result['agent_output']
    if 'reasoning' in ao:
        print(f"\n   Reasoning: {ao['reasoning']}")
    if 'supporting_evidence' in ao:
        print(f"\n   Supporting evidence:")
        for ev in ao['supporting_evidence'][:3]:
            print(f"     • {ev}")
    if 'ruled_out' in ao:
        print(f"\n   Ruled out:")
        for ro in ao['ruled_out']:
            print(f"     • {ro.get('diagnosis', '?')}: {ro.get('reason', 'N/A')}")

    print(f"\n{'─' * 70}")
    if result['lstm_correct'] and result['agent_correct']:
        print("✅ BOTH CORRECT")
    elif not result['lstm_correct'] and result['agent_correct']:
        print("🎉 AGENT IMPROVED")
    elif result['lstm_correct'] and not result['agent_correct']:
        print("⚠️ AGENT HURT")
    else:
        print("❌ BOTH WRONG")

# Inspect first case
if valid_results:
    inspect_case(valid_results[0])


In [ ]:
# Inspect specific case by index
CASE_INDEX = 14

if valid_results and CASE_INDEX < len(valid_results):
    inspect_case(valid_results[CASE_INDEX])

══════════════════════════════════════════════════════════════════════
PATIENT: 19121325
══════════════════════════════════════════════════════════════════════

🎯 TRUE: PANCREATITIS

📊 LSTM TOP-3:
   1. HEART_FAILURE              49.3%  
   2. AFIB_ARRHYTHMIA            18.6%  
   3. SEPSIS                     13.1%  

🤖 AGENT DECISION:
   Chose: SEPSIS
   Confidence: 75.0/100

   Reasoning: The patient presents with tachycardia, tachypnea, and a trend towards hypotension, which are classic signs of sepsis. While the temperature is not elevated, hypothermia can also be a sign of sepsis. The worsening 'H' (likely referring to Hemoglobin or Hematocrit, though not explicitly defined) and 'L' (likely Leukocytes, though not explicitly defined) values, if indicating a falling Hgb/Hct and rising WBC, would further support an infectious process. The absence of clear signs of heart failure or primary arrhythmia makes sepsis the most likely diagnosis.

   Supporting evidence:
     • Heart Rate: 

In [ ]:
print('\n' + '═' * 60)
print('DISAGREEMENTS WHERE TRUE IS IN LSTM TOP-K')
print('═' * 60)

true_in_topk_disagreements = [
    r for r in disagree_cases
    if r['true_label'] in r['lstm_output']['top_k_codes']
]

if true_in_topk_disagreements:
    for r in true_in_topk_disagreements:
        print(f'\n  Patient {r["patient_id"]}')
        print(f'    True: {r["true_label"]}')
        print(f'    LSTM Top-{{TOP_K}}: {r["lstm_output"]["top_k_codes"]}')
        print(f'    Agent: {r["agent_prediction"]}')
else:
    print('  No cases where agent disagreed and true label was in LSTM top-K.')



════════════════════════════════════════════════════════════
DISAGREEMENTS WHERE TRUE IS IN LSTM TOP-K
════════════════════════════════════════════════════════════

  Patient 15372682
    True: SURGICAL_COMPLICATION
    LSTM Top-{TOP_K}: ['ACUTE_MI', 'CORONARY_ARTERY_DZ', 'SURGICAL_COMPLICATION']
    Agent: SURGICAL_COMPLICATION

  Patient 15115846
    True: AFIB_ARRHYTHMIA
    LSTM Top-{TOP_K}: ['AFIB_ARRHYTHMIA', 'TOXIC_METABOLIC', 'STROKE_NEURO']
    Agent: TOXIC_METABOLIC

  Patient 13241979
    True: CORONARY_ARTERY_DZ
    LSTM Top-{TOP_K}: ['CORONARY_ARTERY_DZ', 'ACUTE_MI', 'VALVE_DISEASE']
    Agent: ACUTE_MI

  Patient 11498404
    True: STROKE_NEURO
    LSTM Top-{TOP_K}: ['STROKE_NEURO', 'SEPSIS', 'SURGICAL_COMPLICATION']
    Agent: SURGICAL_COMPLICATION

  Patient 17357689
    True: CORONARY_ARTERY_DZ
    LSTM Top-{TOP_K}: ['CORONARY_ARTERY_DZ', 'ACUTE_MI', 'VALVE_DISEASE']
    Agent: ACUTE_MI

  Patient 16538583
    True: HEART_FAILURE
    LSTM Top-{TOP_K}: ['HEART_FAILURE'

In [ ]:
for result_item in true_in_topk_disagreements:
    inspect_case(result_item)

══════════════════════════════════════════════════════════════════════
PATIENT: 15372682
══════════════════════════════════════════════════════════════════════

🎯 TRUE: SURGICAL_COMPLICATION

📊 LSTM TOP-3:
   1. ACUTE_MI                   25.1%  
   2. CORONARY_ARTERY_DZ         19.5%  
   3. SURGICAL_COMPLICATION      16.7% ✓

🤖 AGENT DECISION:
   Chose: SURGICAL_COMPLICATION
   Confidence: 75.0/100

   Reasoning: The patient's vital signs are generally improving, which is less typical for acute MI or significant CAD decompensation. The worsening PTT and potassium, coupled with a stable but slightly elevated glucose, are non-specific but could be related to a post-surgical inflammatory response or fluid shifts. The absence of specific cardiac markers (like troponin, which is not provided but implied by the differentials) and the general trend of improvement in vitals make acute cardiac events less likely as the primary driver.

   Supporting evidence:
     • Improving vital signs (HR,

---

## 📝 Comparison: Single Agent vs Multi-Agent

| Aspect | Multi-Agent (v2) | Single Agent (v3) |
|--------|------------------|-------------------|
| LLM calls/case | 6 | 1 |
| Cost (40 cases) | ~$0.50 | ~$0.08 |
| Time (40 cases) | ~24 min | ~4 min |
| Context | Fragmented | Complete |
| Reasoning | Structured stages | Single synthesis |

**If this works better:** Scale to 200 cases for robust comparison.